<a href="https://colab.research.google.com/github/kili-technology/kili-python-sdk/blob/main/recipes/file_annotations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# How to attach files to a label


A **file job** is answered with a file rather than a drawing or a piece of text: a render, a
scene file, a report. It is how a step whose output is produced outside Kili -- a 3D tool, a
conversion script -- hands that output back.

In this recipe we will:

- declare two file jobs in a project interface,
- upload a deliverable for each,
- submit one label carrying both,
- read them back and download them.

A file job holds **one** file, the way a transcription holds one text. Several deliverables
means several jobs, all in the same label.


## Setup


In [ ]:
%pip install kili

In [ ]:
from kili.client import Kili

kili = Kili(
    # api_endpoint='https://cloud.kili-technology.com/api/label/v2/graphql',
    # the line above can be uncommented and changed if you are working with an on-premise version of Kili
)

## Creating a project with file jobs


`FILE` declares no categories and no input: its answer is the file itself.


In [ ]:
json_interface = {
    "jobs": {
        "RENDER": {
            "mlTask": "FILE",
            "instruction": "Rendered video",
            "required": 0,
            "isChild": False,
            "content": {},
        },
        "SCENE": {
            "mlTask": "FILE",
            "instruction": "Scene file",
            "required": 0,
            "isChild": False,
            "content": {},
        },
    }
}

project = kili.create_project(
    input_type="IMAGE",
    json_interface=json_interface,
    title="File annotations recipe",
)
project_id = project["id"]
project_id

In [ ]:
kili.append_many_to_dataset(
    project_id=project_id,
    content_array=["https://storage.googleapis.com/label-public-staging/car/car_1.jpg"],
    external_id_array=["car_1"],
)

asset_id = kili.assets(project_id=project_id, fields=["id"])[0]["id"]
asset_id

## Uploading the deliverables


`upload_annotation_file` returns the answer to write into the label: a file job holds one
file, so the returned dictionary is the whole of that job's `jsonResponse`.

The upload does not write a label by itself. That is deliberate: a pipeline producing several
deliverables uploads each, then submits **one** label carrying them all.


In [ ]:
from pathlib import Path

Path("render.mp4").write_bytes(b"not really a video, but enough for the recipe")
Path("scene.blend").write_bytes(b"not really a scene either")

render = kili.upload_annotation_file(
    project_id=project_id, asset_id=asset_id, file_path="render.mp4"
)
scene = kili.upload_annotation_file(
    project_id=project_id, asset_id=asset_id, file_path="scene.blend"
)

render, scene

Note the mime type: `render.mp4` resolves to `video/mp4`, while `.blend` is not a type Python
knows, so it comes back empty and Kili fills it in from the file name.


## Submitting one label carrying both


In [ ]:
kili.append_labels(
    project_id=project_id,
    asset_id_array=[asset_id],
    json_response_array=[{"RENDER": render, "SCENE": scene}],
)

On a **video** project an asset level job is stored under `assetLevel`, so the response would
be `{'assetLevel': {'RENDER': render, 'SCENE': scene}}` instead.


## Reading the files back


In [ ]:
label = kili.labels(project_id=project_id, asset_id=asset_id, fields=["jsonResponse"])[-1]
label["jsonResponse"]

Each job names its file by id. The id is resolved into bytes on demand:


In [ ]:
for job_name, answer in label["jsonResponse"].items():
    path = kili.download_annotation_file(
        project_id=project_id,
        asset_id=asset_id,
        file_id=answer["fileId"],
        output_path=answer["fileName"],
    )
    print(f"{job_name}: {path} ({answer['fileMimeType'] or 'type decided by Kili'})")

The download url is signed when asked for and is short lived, so it is fetched per call
rather than stored -- no url is ever persisted in a label.


## Cleanup


In [ ]:
kili.delete_project(project_id=project_id)

Path("render.mp4").unlink(missing_ok=True)
Path("scene.blend").unlink(missing_ok=True)